<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/ML1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, cohen_kappa_score
from scipy.stats import pearsonr

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving training_set_rel3.tsv.zip to training_set_rel3.tsv.zip


In [ ]:
!unzip training_set_rel3.tsv.zip

Archive:  training_set_rel3.tsv.zip
  inflating: training_set_rel3.tsv   


In [ ]:
import pandas as pd

df = pd.read_csv('training_set_rel3.tsv', sep='\t', encoding='latin1')

In [ ]:
# Filter only essay set 1
set1 = df[df['essay_set'] == 1]

# check shape
set1.shape

(1783, 28)

In [ ]:
set1 = set1[['essay_id','essay','domain1_score']]

set1.head()

,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [ ]:

set1['essay'] = set1['essay'].str.replace('\n',' ')
set1['essay'] = set1['essay'].str.strip()
set1.head() ##removes line breaks inside each essay.

,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [ ]:
X = set1['essay']
y = set1['domain1_score']

In [ ]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    set1['essay_id'],
    test_size=0.2,
    random_state=42
)

In [ ]:
ridge_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        stop_words='english'
    )),
    ('model', Ridge(alpha=1.0))
])

In [ ]:
ridge_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('model', Ridge())])

In [ ]:
y_pred_continuous = ridge_pipeline.predict(X_test)

y_pred_rounded = np.rint(y_pred_continuous)   # round to nearest integer
y_pred_rounded = np.clip(y_pred_rounded, 2, 12)  # keep within valid score range
y_pred_rounded = y_pred_rounded.astype(int)

In [ ]:
qwk = cohen_kappa_score(y_test, y_pred_rounded, weights='quadratic')
pcc, _ = pearsonr(y_test, y_pred_rounded)
mae = mean_absolute_error(y_test, y_pred_rounded)

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

QWK: 0.6392177711362722
PCC: 0.7059202959856586
MAE: 0.7871148459383753


In [ ]:
results_df = pd.DataFrame({
    'essay_id': id_test.values,
    'essay_text': X_test.values,
    'human_score': y_test.values,
    'predicted_score': y_pred_rounded
})

results_df['difference'] = results_df['predicted_score'] - results_df['human_score']

results_df.head(10)

,essay_id,essay_text,human_score,predicted_score,difference
0,827,I think computers have a postitive affect on p...,6,8,2
1,1477,I blive that computers have a lot of effects o...,6,8,2
2,234,Many people think that computers are not a goo...,8,9,1
3,801,"Dear Newspaper people, @CAPS1 you might heard ...",9,8,-1
4,780,More and more people are using computers on a ...,9,9,0
5,271,"Dear Local Newspaper, I've heard about your pr...",8,8,0
6,419,The local newspaper wants to know my opinion o...,6,7,1
7,1441,"Dear local nepaper, I have reacently heard abo...",8,8,0
8,1391,"Dear @CAPS1 of the newspaper, I understand you...",8,9,1
9,112,"Dear @PERSON1, @CAPS1 these days are telling k...",9,9,0
